In [ ]:
# GARBAGE RECOGNIZER PROJECT

#   IMPORT DATA

# Kaggle authentication
# Before running this part of te code, generate a kaggle.json file from:
# Kaggle -> Settings -> API -> Create Legacy API Key
# Upload the downloaded kaggle.json file to the Colab working directory
# The notebook will use the file to authenticate and automatically download the
# dataset


!pip install -q kaggle



!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d sumn2u/garbage-classification-v2

!unzip -q garbage-classification-v2.zip -d dataset


# check that the dataset has been correctly downloaded

import os

print(os.listdir("dataset"))


# check dataset structure and folders

!find dataset -maxdepth 2 -type d | sort

# check the image number in each version

from pathlib import Path

for version in ["original", "standardized_256", "standardized_384"]:
    n = len(list(Path(f"dataset/{version}").rglob("*.jpg")))
    print(version, n)

# check image size in standardized_256 folder


from pathlib import Path
from PIL import Image

sizes = set()

for img_path in Path("dataset/standardized_256").rglob("*.jpg"):
    with Image.open(img_path) as img:
        sizes.add(img.size)

print(sizes)


# Set work directory

from pathlib import Path

DATASET_DIR = Path("dataset/standardized_256")







#    DATASET AUDIT


# class distribution

from pathlib import Path

class_counts = {}

for class_dir in DATASET_DIR.iterdir():
    if class_dir.is_dir():
        class_counts[class_dir.name] = len(list(class_dir.glob("*.jpg")))

for cls, count in sorted(class_counts.items()):
    print(f"{cls:12s}: {count}")

# Histogram

class_counts

import matplotlib.pyplot as plt

classes = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(10,5))
plt.bar(classes, counts)

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()


# imbalance ratio

imbalance_ratio = max(class_counts.values()) / min(class_counts.values())

print("Imbalance ratio:", round(imbalance_ratio, 2))



# corrupted images

from PIL import Image

corrupted = []

for img_path in DATASET_DIR.rglob("*.jpg"):
    try:
        with Image.open(img_path) as img:
            img.verify()
    except Exception:
        corrupted.append(img_path)

print("Corrupted files:", len(corrupted))


# duplicated images

from pathlib import Path
import hashlib

hash_dict = {}
duplicates = []

for img_path in DATASET_DIR.rglob("*.jpg"):

    with open(img_path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    if file_hash in hash_dict:
        duplicates.append((img_path, hash_dict[file_hash]))
    else:
        hash_dict[file_hash] = img_path

print("Duplicate images:", len(duplicates))







#      STRATIFIED SUBSAMPLE


# build the dataset in tabellar form ( list of images and labels)

from pathlib import Path
import pandas as pd

DATASET_DIR = Path("dataset/standardized_256")


data = []

for class_dir in DATASET_DIR.iterdir():
    if class_dir.is_dir():
        for img_path in class_dir.glob("*.jpg"):
            data.append((str(img_path), class_dir.name))

df = pd.DataFrame(data, columns=["path", "label"])

print(df.head())
print("Total images:", len(df))

# subsample


from sklearn.model_selection import train_test_split

df_subsample, _ = train_test_split(
    df,
    test_size=0.75,
    stratify=df["label"],
    random_state=42
)

print("Subsample size:", len(df_subsample))


# check that each label has the same proportion it had before

print(df_subsample["label"].value_counts(normalize=True))
print(df["label"].value_counts(normalize=True))

# build the graph and the plot for the report

import pandas as pd

df_full = df["label"].value_counts(normalize=True)
df_sub = df_subsample["label"].value_counts(normalize=True)

table = pd.DataFrame({
    "full": df_full,
    "subsample": df_sub
})

table = table.round(4).sort_values("full", ascending=False)

table

# plot

import matplotlib.pyplot as plt

orig = df["label"].value_counts().sort_index()
sub = df_subsample["label"].value_counts().sort_index()

x = orig.index

plt.figure(figsize=(10,5))
plt.bar(x, orig.values, alpha=0.5, label="Original")
plt.bar(x, sub.values, alpha=0.5, label="Subsample (25%)")

plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()






# STRATIFIED TRAIN TEST SPLIT


from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_subsample,
    test_size=0.2,
    stratify=df_subsample["label"],
    random_state=42
)


# check number of images in train and test sets

print("Train:", len(train_df))
print("Test:", len(test_df))

#check class distribution between train and test

print(train_df["label"].value_counts(normalize=True))
print(test_df["label"].value_counts(normalize=True))

# build the graph

train_dist = train_df["label"].value_counts(normalize=True)
test_dist = test_df["label"].value_counts(normalize=True)

split_table = pd.DataFrame({
    "train": train_dist,
    "test": test_dist
})

split_table["diff"] = (split_table["train"] - split_table["test"]).abs()

split_table = split_table.round(4).sort_values("train", ascending=False)

split_table


# divide the trining set in 3 folds

from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# create the trin validation structure with the folds


for fold, (train_idx, val_idx) in enumerate(
        skf.split(train_df, train_df["label"]),
        start=1):

    train_fold = train_df.iloc[train_idx]
    val_fold = train_df.iloc[val_idx]

    print(f"\nFold {fold}")
    print("Train:", len(train_fold))
    print("Validation:", len(val_fold))

    print(
        train_fold["label"]
        .value_counts(normalize=True)
        .round(4)
    )

    print(
        val_fold["label"]
        .value_counts(normalize=True)
        .round(4)
    )


# save the folds


folds = []

for train_idx, val_idx in skf.split(
        train_df,
        train_df["label"]):

    folds.append((train_idx, val_idx))







# DEFINITIONS


IMG_SIZE = (256, 256)
NUM_CLASSES = train_df["label"].nunique()

SEED = 42
EPOCHS = 20
AUTOTUNE = tf.data.AUTOTUNE






# NORMALIZATION AND DATA AUGMENTATION



# Normalization ( pixel rescaling)

import tensorflow as tf
from tensorflow.keras import layers

rescaling = tf.keras.Sequential([
    layers.Rescaling(1./255)
])



# data augmentation

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

# Early stopping

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)



# HYPERPARAMETHERS TUNING VIA RANDOM SEARCH WITH 3 FOLD CV



# hyperparameters space


dropout_values = [0.3, 0.5]
learning_rates = [1e-3, 5e-4, 1e-4]
batch_sizes = [16, 32]
dense_units = [64, 128]



# network structure

from tensorflow.keras.optimizers import Adam

def build_model(
        dropout_rate=0.3,
        dense_units=128,
        learning_rate=1e-3):

    model = keras.Sequential([

        layers.Input(shape=(256, 256, 3)),

        # preprocessing
        rescaling,
        data_augmentation,

        # Block 1
        layers.Conv2D(
            32,
            (3, 3),
            padding="same",
            use_bias=False
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),

        # Block 2
        layers.Conv2D(
            64,
            (3, 3),
            padding="same",
            use_bias=False
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),

        # Block 3
        layers.Conv2D(
            128,
            (3, 3),
            padding="same",
            use_bias=False
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),

        # classifier
        layers.GlobalAveragePooling2D(),

        layers.Dense(
            dense_units,
            activation="relu"
        ),

        layers.Dropout(dropout_rate),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])

    optimizer = Adam(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# definitions

kernel_size = 3
activation = "relu"
optimizer = tf.keras.optimizers.Adam

# net structure

from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([

    layers.Input(shape=(256, 256, 3)),

    # preprocessing
    rescaling,
    data_augmentation,

    # Block 1
    layers.Conv2D(32, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),

    # classifier
    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()


early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)



# retrain

model = tf.keras.Sequential([
    layers.Input(shape=(256,256,3)),
    rescaling,
    data_augmentation,
    ...
])

# final test, precision, F1, recall

# sanity test, lable reshuffle
